In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parent))

# Test — Training loop (`training/negative_sampling.py`, `training/train.py`)

Complements `test_hgnn.ipynb` (which checks model *correctness*: shapes, no leakage, gradients flow). This notebook checks training *mechanics*: negative sampling is valid, the loop actually reduces loss, checkpoints are saved and reloadable, and batching doesn't change correctness.

In [2]:
import torch

from models.hgnn import DrugSafetyHGNN
from models.encoder import split_polypharmacy_edges
from training.negative_sampling import corrupt_edges
from training.train import train_one_epoch, evaluate, run

PROJECT_ROOT = Path("..")

GRAPH_DIR = PROJECT_ROOT / "graph"

data = torch.load(GRAPH_DIR / "heterodata_with_features.pt", weights_only=False)

ddi_key = ("drug", "polypharmacy", "drug")
num_drugs = data["drug"].num_nodes
num_relations = int(data[ddi_key].edge_type.max().item()) + 1
print("num_drugs:", num_drugs, "| num_relations:", num_relations)


c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_drugs: 70 | num_relations: 1301


## Test 1 — negative sampling preserves relation type, changes exactly one endpoint

In [3]:
gen = torch.Generator().manual_seed(0)

edge_index = data[ddi_key].edge_index[:, :20]
edge_type = data[ddi_key].edge_type[:20]

neg_edge_index, neg_edge_type = corrupt_edges(edge_index, edge_type, num_drugs, generator=gen)

# relation type must be unchanged
assert torch.equal(neg_edge_type, edge_type), "corruption must not change the relation"

# indices must be valid drug node ids
assert neg_edge_index.min() >= 0 and neg_edge_index.max() < num_drugs

# for each edge, exactly one endpoint should differ from the positive
# (the other stays fixed -- that's what "corrupt src OR dst" means)
src_changed = neg_edge_index[0] != edge_index[0]
dst_changed = neg_edge_index[1] != edge_index[1]
both_or_neither_changed = (src_changed & dst_changed) | (~src_changed & ~dst_changed)

# a small number of "neither changed" cases are expected -- the random
# replacement node can coincidentally equal the original by chance
n_suspicious = both_or_neither_changed.sum().item()
print(f"edges where both/neither endpoint changed (chance collisions expected): {n_suspicious}/20")
assert n_suspicious < 20, "corruption does not appear to be doing anything"
print("PASS")


edges where both/neither endpoint changed (chance collisions expected): 0/20
PASS


## Test 2 — one training step produces a finite loss and updates parameters

In [4]:
torch.manual_seed(1)
model = DrugSafetyHGNN(data, num_relations=num_relations)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
gen = torch.Generator().manual_seed(1)

splits = split_polypharmacy_edges(data, val_frac=0.1, test_frac=0.1, seed=42)

params_before = [p.clone() for p in model.parameters()]

loss = train_one_epoch(
    model, data,
    splits["train"]["edge_index"], splits["train"]["edge_type"],
    optimizer, num_drugs, batch_size=min(100000, len(splits["train"]["edge_type"])), generator=gen,
)
 
assert torch.isfinite(torch.tensor(loss)), f"loss is not finite: {loss}"
print("loss:", loss)

changed = any(
    not torch.equal(before, after)
    for before, after in zip(params_before, model.parameters())
)
assert changed, "no parameters changed after one training step" 
print("PASS -- loss finite, parameters updated")


loss: 0.6931540029389518
PASS -- loss finite, parameters updated


## Test 3 — evaluate() returns valid AUROC/AUPRC in [0, 1]

In [5]:
gen = torch.Generator().manual_seed(2)
auroc, auprc = evaluate(
    model, data, splits["val"]["edge_index"], splits["val"]["edge_type"], num_drugs, gen
)
print("val AUROC:", auroc, "| val AUPRC:", auprc)
assert 0.0 <= auroc <= 1.0
assert 0.0 <= auprc <= 1.0
print("PASS")


val AUROC: 0.5064195553803925 | val AUPRC: 0.5042995307831705
PASS


## Test 4 — loss trend is real, not noise

In [6]:
torch.manual_seed(3)
model2 = DrugSafetyHGNN(data, num_relations=num_relations)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
gen2 = torch.Generator().manual_seed(3)

losses = []
for epoch in range(10):
    loss = train_one_epoch(
        model2, data,
        splits["train"]["edge_index"], splits["train"]["edge_type"],
        optimizer2, num_drugs, batch_size=min(100000, len(splits["train"]["edge_type"])), generator=gen2,
    )
    losses.append(loss)

print("losses:", [round(l, 4) for l in losses])
assert losses[-1] < losses[0], "loss did not decrease over 10 epochs"
print("PASS -- loss decreased from", round(losses[0], 4), "to", round(losses[-1], 4))


losses: [0.6932, 0.693, 0.6929, 0.6927, 0.6921, 0.6915, 0.6909, 0.6902, 0.6895, 0.6886]
PASS -- loss decreased from 0.6932 to 0.6886


## Test 5 — full run() end-to-end: checkpoints saved and reloadable

In [9]:
import tempfile
import json

with tempfile.TemporaryDirectory() as tmpdir:
    model3, history = run(
        data_path=str(GRAPH_DIR / "heterodata_with_features.pt"),
        output_dir=tmpdir,
        epochs=3,
        batch_size=min(50000, len(splits["train"]["edge_type"])),
        lr=1e-3,
        seed=7,
    )

    tmpdir_path = Path(tmpdir)
    assert (tmpdir_path / "best_model.pt").exists()
    assert (tmpdir_path / "final_model.pt").exists()
    assert (tmpdir_path / "training_history.json").exists()

    history_reloaded = json.loads((tmpdir_path / "training_history.json").read_text())
    assert len(history_reloaded) == 3

    # confirm the checkpoint actually reloads into a fresh model of the same shape
    reload_model = DrugSafetyHGNN(data, num_relations=num_relations)
    state_dict = torch.load(tmpdir_path / "final_model.pt", weights_only=True)
    reload_model.load_state_dict(state_dict)

print("PASS -- checkpoints saved, history logged, state_dict reloads cleanly")


{'epoch': 1, 'train_loss': 0.693123459815979, 'lr': 0.001, 'val_auroc': 0.509430554932336, 'val_auprc': 0.5071971254664939}
{'epoch': 2, 'train_loss': 0.692919162603525, 'lr': 0.001, 'val_auroc': 0.515945564456449, 'val_auprc': 0.5131416871159118}
{'epoch': 3, 'train_loss': 0.6922764548888574, 'lr': 0.001, 'val_auroc': 0.5259440460891616, 'val_auprc': 0.5200437212000039}
Final test AUROC: 0.5257, AUPRC: 0.5199
PASS -- checkpoints saved, history logged, state_dict reloads cleanly
